# P-0: free open-source video and voice probe

Goal: find out, with no Magic Hour key and no credits, whether free open-source models on a free Colab GPU can make usable vertical (9:16) shots for the microdrama pipeline.

What it does:
1. Reports the GPU.
2. Generates one short 9:16 clip with LTX-Video (Apache-2.0 code, via diffusers) and logs time and peak GPU memory.
3. Generates one line of English speech with Kokoro (Apache-2.0) and logs time.
4. Writes `p0_results.json` and the sample files so the numbers can be read and the clip judged by eye.

Run with a GPU runtime (Runtime, Change runtime type, T4 GPU). Run All. Nothing here is paid and no key is needed. The first run downloads model weights from Hugging Face, which can take several minutes.

If the GPU runs out of memory, the notebook records that as a result instead of crashing. That result is useful: it means a free T4 cannot carry this model at this size.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "sentencepiece", "imageio", "imageio-ffmpeg", "kokoro", "soundfile"], check=True)

In [ ]:
import json, time, os, torch
results = {"gpu": None, "video": {}, "tts": {}}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    results["gpu"] = {"name": props.name, "vram_gb": round(props.total_memory / 1e9, 1), "bf16": torch.cuda.is_bf16_supported()}
print(results["gpu"])

## Video probe

Shot: a 9:16 portrait, 480 wide by 704 high, about 3 seconds. The prompt is a typical microdrama hook shot. The model is `Lightricks/LTX-Video` loaded through diffusers with CPU offload and tiled decoding to fit a small GPU.

In [ ]:
import gc
from diffusers import LTXPipeline
from diffusers.utils import export_to_video
dtype = torch.bfloat16 if results["gpu"] and results["gpu"]["bf16"] else torch.float16
prompt = "Vertical cinematic close-up of a young woman in a rain-soaked cafe window, she reads a text message on her phone and her face freezes in shock, neon reflections, shallow depth of field, realistic, steady camera"
negative = "worst quality, blurry, distorted face, extra fingers, jittery motion, watermark, text"
t0 = time.time()
try:
    pipe = LTXPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=dtype)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    results["video"]["load_s"] = round(time.time() - t0, 1)
    torch.cuda.reset_peak_memory_stats()
    t1 = time.time()
    out = pipe(prompt=prompt, negative_prompt=negative, width=480, height=704, num_frames=73, num_inference_steps=30, guidance_scale=3.0, generator=torch.Generator("cpu").manual_seed(7))
    results["video"]["gen_s"] = round(time.time() - t1, 1)
    results["video"]["peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
    export_to_video(out.frames[0], "p0_clip.mp4", fps=24)
    results["video"]["status"] = "ok"
    results["video"]["seconds_of_video"] = round(73 / 24, 1)
except torch.cuda.OutOfMemoryError as e:
    results["video"]["status"] = "out_of_memory"
except Exception as e:
    results["video"]["status"] = "error"
    results["video"]["error"] = str(e)[:400]
finally:
    gc.collect()
    torch.cuda.empty_cache()
print(results["video"])

## Voice probe

One English line with Kokoro, American English voice `af_heart`. Kokoro needs the `espeak-ng` system package for some words, so it is installed first.

In [ ]:
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=False)
import soundfile as sf, numpy as np
t0 = time.time()
try:
    from kokoro import KPipeline
    tts = KPipeline(lang_code="a")
    chunks = [np.asarray(a) for _, _, a in tts("You told me it was over. So why is his ring on your finger?", voice="af_heart")]
    audio = np.concatenate(chunks)
    sf.write("p0_voice.wav", audio, 24000)
    results["tts"] = {"status": "ok", "gen_s": round(time.time() - t0, 1), "audio_s": round(len(audio) / 24000, 1)}
except Exception as e:
    results["tts"] = {"status": "error", "error": str(e)[:400]}
print(results["tts"])

In [ ]:
with open("p0_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

In [ ]:
from IPython.display import Video, Audio, display
if os.path.exists("p0_clip.mp4"):
    display(Video("p0_clip.mp4", embed=True, width=240))
if os.path.exists("p0_voice.wav"):
    display(Audio("p0_voice.wav"))

## How to read the result

- `video.status` is `ok`: note `gen_s` for a 3 second clip. A 6 second shot is roughly double. A 10 shot episode of 6 second shots is therefore about `gen_s x 2 x 10`, before any rerolls.
- Look at the clip. Does the face stay stable, is the motion natural, is it watchable at phone size? That is the real test, not the timing.
- `out_of_memory`: this free GPU is too small for this model at this size. Next try is a smaller or quantised model.
- `tts.status` is `ok`: listen to `p0_voice.wav` and judge whether it is good enough for dialogue.

Send the printed JSON and a screenshot of the clip back to the build thread.